# Starter kit — Hackathon Mines Paris 2026 — Analyse d'incidents PV / éolien

## Objectif du hackathon

Construire un pipeline agentique qui, à partir des rapports d'incidents PV et éolien `data/incidents_students.csv`, réalise :
- du **topic modeling** (regroupement des incidents par thème),
- une **analyse de causes**,
- des **statistiques globales** (par site, sévérité, saisonalité, durée d'arrêt, coût...),
- des **propositions de remédiation** (maintenance, évolutions techniques).

Ce notebook couvre la première brique, le topic modeling, et sert de base pour construire le reste
du pipeline.

## 1. Configuration

In [1]:
import os

# Réseau d'entreprise avec inspection SSL : utiliser le magasin de certificats de l'OS
# (doit être appelé avant la création de tout client HTTP).
import truststore

truststore.inject_into_ssl()

from dotenv import load_dotenv

# override=True : le .env prime sur d'éventuelles variables déjà présentes dans le kernel
load_dotenv(override=True)

AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]
EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
LLM_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

# Vérification de la config lue (la clé n'est jamais affichée)
print("endpoint   :", repr(AZURE_OPENAI_ENDPOINT))
print("api_version:", repr(AZURE_OPENAI_API_VERSION))
print("embedding  :", repr(EMBEDDING_DEPLOYMENT))
print("chat       :", repr(LLM_DEPLOYMENT))
if "/api/projects" in AZURE_OPENAI_ENDPOINT:
    print(
        "⚠️ L'endpoint contient /api/projects : AzureOpenAI attend l'endpoint de la ressource, "
        "ex. https://<ressource>.cognitiveservices.azure.com/"
    )

endpoint   : 'https://hackathon-mines-paris-2026-foundry-resource.services.ai.azure.com/'
api_version: '2024-12-01-preview'
embedding  : 'text-embedding-3-small-3-uh79pz'
chat       : 'gpt-4.1-nano-1'


In [2]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Test de connexion : isole un problème de config (endpoint / version / déploiement)
# avant de lancer le calcul des embeddings sur tout le jeu de données.
_probe = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=["test"])
print("Connexion OK, dimension des embeddings :", len(_probe.data[0].embedding))

Connexion OK, dimension des embeddings : 1536


## 2. Charger les données

Les rapports sont fournis dans 3 formats (`document_format`) :
- `text` : texte déjà disponible dans la colonne `text` du CSV.
- `pdf_digital` : PDF avec texte extractible (`document_path` pointe vers `data/pdf_digital/`).
- `pdf_scan` : PDF scanné (image), nécessitant de l'OCR (`document_path` pointe vers
  `data/pdf_scan/`).

Pour ces deux derniers formats, seules les colonnes `site`, `document_format` et `document_path`
sont renseignées dans le CSV : à vous d'extraire le texte (et le reste des métadonnées) des PDF.

Ce starter kit se limite volontairement au sous-ensemble `text`, déjà prêt à l'emploi, pour
démarrer le topic modeling. 

In [3]:
import pandas as pd

df = pd.read_csv("data/incidents_students.csv")
print(df["document_format"].value_counts())

df = df[df["document_format"] == "text"].reset_index(drop=True)
print(df.shape)
df.head()

document_format
pdf_scan       600
text           600
pdf_digital    600
Name: count, dtype: int64
(600, 13)


,incident_id,site,asset_type,equipment_id,date,severity,downtime_hours,estimated_cost_eur,weather_condition,technician_id,text,document_format,document_path
0,2,Parc éolien Sud-1,wind,SUD1-WTG09,2023-12-13,modérée,9.7,7069.41,vent fort / tempête,TECH-SUD1-3,"En début d'après-midi, lors de l'inspection vi...",text,NaN
1,3,Centrale PV Est-2,solar,EST2-INV05,2023-08-27,mineure,1.0,261.57,temps calme,TECH-EST2-3,"En début d’après-midi, le système de supervisi...",text,NaN
2,4,Centrale PV Est-2,solar,EST2-INV01,2023-09-06,mineure,1.7,414.83,vent modéré,TECH-EST2-3,Lors d’un contrôle périodique sur la centrale ...,text,NaN
3,8,Centrale PV Est-2,solar,EST2-INV01,2023-07-05,modérée,14.8,5460.50,conditions nominales,TECH-EST2-2,"Vers la fin de l’après-midi, une alarme incend...",text,NaN
4,9,Centrale PV Est-2,solar,EST2-INV08,2024-06-22,modérée,4.7,1672.96,temps calme,TECH-EST2-2,"En début d'après-midi, une alarme a été déclen...",text,NaN


In [4]:
df["asset_type"].value_counts(), df["severity"].value_counts()

(asset_type
 wind     300
 solar    300
 Name: count, dtype: int64,
 severity
 mineure    368
 modérée    182
 sévère      50
 Name: count, dtype: int64)

In [5]:
for text in df["text"].sample(3, random_state=0):
    print(text)
    print("-" * 80)

En début d’après-midi, une alerte incendie a été déclenchée au sein du bâtiment support de la centrale photovoltaïque, localisé dans la zone centrale de l’installation. L’incendie, qui s’est déclaré sur une partie de la toiture et des panneaux photovoltaïques, a rapidement été identifié par le système de détection incendie. Les pompiers ont été dépêchés sur place dès réception de l’alerte, et leur intervention a duré plusieurs heures afin de maîtriser le feu. Le personnel de la centrale a été informé en temps réel par le système de supervision, et une équipe de maintenance a sécurisé la zone après intervention. L’incendie a été contenu, limitant ainsi les dommages à une zone restreinte du bâtiment et des panneaux photovoltaïques.
--------------------------------------------------------------------------------
Le matin, une élévation de température a été détectée localement sur un panneau photovoltaïque dans la zone sud de la centrale. L’alerte a été générée par le système de supervisio

## 3. Calculer les embeddings du texte

Un embedding représente un texte par un vecteur tel que deux textes similaires ont des vecteurs
proches. Calcul via Azure OpenAI, par lots, avec mise en cache locale (`.npy`) pour éviter de
refaire les appels à chaque exécution.

In [6]:
import math
import re
import time

import numpy as np
from openai import RateLimitError
from tqdm import tqdm


def wait_with_progress(seconds: int, desc: str) -> None:
    """Attente avec barre de progression (décompte en secondes)."""
    for _ in tqdm(range(seconds), desc=desc, unit="s", leave=False):
        time.sleep(1)


def embed_batch_with_retry(batch: list[str], max_attempts: int = 10):
    """Appelle l'API d'embeddings ; sur erreur 429, attend le délai indiqué par Azure puis réessaie.

    `max_retries=0` : on désactive les tentatives silencieuses du SDK pour que l'attente
    soit gérée ici, avec une barre de progression visible.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            return client.with_options(max_retries=0).embeddings.create(
                model=EMBEDDING_DEPLOYMENT, input=batch
            )
        except RateLimitError as e:
            if attempt == max_attempts:
                raise
            match = re.search(r"retry after (\d+) seconds", str(e))
            wait = int(match.group(1)) + 1 if match else 30
            wait_with_progress(wait, f"Quota embeddings atteint (429), tentative {attempt}/{max_attempts}")


def azure_embed_documents(
    docs: list[str], batch_size: int = 16, partial_path: str = "embeddings_students.partial.npy"
) -> np.ndarray:
    """Calcule les embeddings par lots ; la progression est sauvegardée pour reprendre après interruption."""
    all_embeddings = list(np.load(partial_path)) if os.path.exists(partial_path) else []
    start = len(all_embeddings) // batch_size  # on ne garde que des lots complets
    all_embeddings = all_embeddings[: start * batch_size]
    n_batches = math.ceil(len(docs) / batch_size)

    for i in tqdm(range(start, n_batches), desc="Embeddings", unit="lot", initial=start, total=n_batches):
        batch = docs[i * batch_size : (i + 1) * batch_size]
        response = embed_batch_with_retry(batch)
        all_embeddings.extend(item.embedding for item in response.data)
        if (i + 1) % 10 == 0:
            np.save(partial_path, np.array(all_embeddings))

    if os.path.exists(partial_path):
        os.remove(partial_path)
    return np.array(all_embeddings)


embeddings_path = "embeddings_students.npy"
embeddings = None
if os.path.exists(embeddings_path):
    cached = np.load(embeddings_path)
    if cached.ndim == 2 and cached.shape[0] == len(df):
        embeddings = cached
    else:
        # Cache incohérent (calcul partiel, autre jeu de données...) : on le recalcule
        print(f"Cache ignoré : forme {cached.shape}, attendu ({len(df)}, dim)")

if embeddings is None:
    embeddings = azure_embed_documents(df["text"].tolist())
    np.save(embeddings_path, embeddings)

assert embeddings.shape[0] == len(df), f"{embeddings.shape[0]} embeddings pour {len(df)} documents"
embeddings.shape

Embeddings: 100%|██████████| 38/38 [01:04<00:00,  1.70s/lot]


(600, 1536)

## 4. Topic modeling avec BERTopic

- `CountVectorizer` : extrait les mots-clés de chaque topic une fois les clusters formés
  (stopwords français, tokens numériques exclus).
- `UMAP` : réduit la dimension des embeddings avant clustering.
- `HDBSCAN` : forme les clusters ; les documents isolés sont marqués `-1` (non classés).
- `min_cluster_size` : taille minimale d'un topic.

In [7]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

FRENCH_STOPWORDS = [
    "le",
    "la",
    "les",
    "un",
    "une",
    "des",
    "de",
    "du",
    "et",
    "en",
    "sur",
    "au",
    "aux",
    "pour",
    "par",
    "est",
    "sont",
    "a",
    "été",
    "suite",
    "lors",
    "dans",
    "que",
    "qui",
    "ce",
    "cette",
    "ces",
    "afin",
    "plus",
    "après",
    "avant",
]

umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words=FRENCH_STOPWORDS,
    token_pattern=r"(?u)\b[^\W\d_][^\W\d_]+\b",
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(df["text"].tolist(), embeddings=embeddings)
df["topic"] = topics
print(f"{len(set(topics)) - (1 if -1 in topics else 0)} topics trouvés")

2026-10-09 12:28:49,568 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-09 12:28:54,958 - BERTopic - Dimensionality - Completed ✓
2026-10-09 12:28:54,958 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-09 12:28:55,035 - BERTopic - Cluster - Completed ✓
2026-10-09 12:28:55,037 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-09 12:28:55,075 - BERTopic - Representation - Completed ✓


20 topics trouvés


## 5. Explorer les résultats

In [8]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,9,-1_code_freinage_défaut_système,"[code, freinage, défaut, système, mécanique, r...","[En début d'après-midi, la supervision du parc..."
1,0,101,0_vent_sécurité_automatique_arrêt,"[vent, sécurité, automatique, arrêt, vitesse, ...","[Lors d'une journée venteuse, l'anémomètre du ..."
2,1,44,1_feu_incendie_propagation_extincteur,"[feu, incendie, propagation, extincteur, dépar...","[Vers la mi-journée, une alarme incendie s’est..."
3,2,42,2_prochaine_mécanisme_routine_planifiée,"[prochaine, mécanisme, routine, planifiée, cor...","[Lors d'une inspection de routine, une anomali..."
4,3,41,3_chaud_point_boîtier_jonction,"[chaud, point, boîtier, jonction, thermographi...",[Lors d'une inspection thermique programmée su...
5,4,40,4_isolement_câblage_périodique_anomalie,"[isolement, câblage, périodique, anomalie, cor...",[Lors d’un contrôle périodique effectué dans l...
6,5,36,5_pale_vibrations_extrême_cosmétique,"[pale, vibrations, extrême, cosmétique, anorma...","[En début d'après-midi, lors de l'inspection v..."
7,6,32,6_incendie_bâtiment_toiture_panneaux,"[incendie, bâtiment, toiture, panneaux, photov...","[Vers la mi-journée, un incendie s’est déclaré..."
8,7,32,7_pale_rupture_fissure_micropieux,"[pale, rupture, fissure, micropieux, éolienne,...","[Vers la fin de l’après-midi, une alerte a été..."
9,8,26,8_jonction_boîtier_connecteur_préventif,"[jonction, boîtier, connecteur, préventif, dys...","[En début d'après-midi, une alarme a été décle..."


In [9]:
topic_model.visualize_barchart(top_n_topics=12)

In [10]:
topic_model.visualize_hierarchy()

Les mots-clés seuls suffisent rarement à comprendre un topic : regarde aussi
quelques documents représentatifs.

In [11]:
def show_topic_examples(topic_id: int, n: int = 3) -> None:
    print("Mots-clés :", [word for word, _ in topic_model.get_topic(topic_id)])
    examples = df.loc[df["topic"] == topic_id, "text"].head(n)
    for text in examples:
        print("-", text)


show_topic_examples(0)

Mots-clés : ['vent', 'sécurité', 'automatique', 'arrêt', 'vitesse', 'tempête', 'redémarrage', 'système', 'turbine', 'parc']
- En début d'après-midi, le système de contrôle a déclenché l'arrêt automatique de sécurité d'une éolienne située dans la zone dominante des vents forts. L'alerte a été immédiatement détectée par la supervision, qui a enregistré une montée rapide de la vitesse du vent dépassant la limite paramétrée. Un technicien a été dépêché sur place pour vérifier l’état de la machine et s’assurer qu’elle ne présente pas de dommages apparents. Après la fin de la tempête et la baisse du vent, le système de supervision a confirmé que la vitesse était revenue dans la plage normale, permettant ainsi le redémarrage automatique de l’éolienne. La machine a repris son fonctionnement en mode normal, sans intervention manuelle supplémentaire. Aucune anomalie majeure n’a été constatée lors de cette opération.
- Lors d'une journée venteuse, le système de sécurité du parc éolien a détecté u

## 6. Pistes pour aller plus loin

0. **Sauvegarder vos générations.** Pensez à sauvegarder vos générations pour éviter de perdre votre 
   travail et consommer des ressources LLM supplémentaires.
1. **Rajouter les docs PDF.** Lire et inclure les documents au format PDF digitaux et scans.
2. **Caractériser les topics.** Nommer les différents topics identifier et les analyser au mieux
    pour en faire ressortir les informations associées.
3. **Plan de remédiation.** Rédiger un plan de remédiation associé avec un agent
   ou un système d'agents en charge de differents aspects.
4. **Traiter les rapports `pdf_digital` et `pdf_scan`.** Ce starter kit ne couvre que le
   sous-ensemble `text` (600/1800 rapports).
5. **Séparer PV et éolien avant de clusteriser.** Le vocabulaire des deux domaines diffère
   fortement ; les mélanger risque de faire ressortir cette séparation avant les vraies causes.
6. **Ajuster `min_cluster_size`.** Plus petit → plus de topics, plus fins ; plus grand → moins de
   topics, plus larges.
7. **Ajuster `ngram_range`** pour capter les expressions à plusieurs mots.
8. **Fusionner les topics redondants** avec `topic_model.reduce_topics(...)`, guidé par
   `visualize_hierarchy()`.
9. **Croiser les topics avec les métadonnées** (`site`, `date`, `severity`, `downtime_hours`,
   `estimated_cost_eur`, `weather_condition`) pour enrichir l'interprétation.
10. **Approfondir avec une pipeline agentique.** Les deux appels LLM ci-dessus (nom + analyse)
   sont figés en un seul passage par topic. Un agent outillé (accès au dataframe, aux fonctions
   ci-dessus, à une recherche web) peut itérer pour approfondir l'analyse et croiser plusieurs
   topics entre eux, vérifier une hypothèse sur les métadonnées, ou creuser un signal faible avant
   de conclure, plutôt qu'une synthèse en un seul coup.
   